In [4]:
import sys
import os

from pathlib import Path

from dotenv import load_dotenv
from pyspark.sql import SparkSession

load_dotenv()

sys.path.append("../../")
from src.lib.mlflow import MlflowHandler
from src.main import _ensure_java_home
mlflow_handler = MlflowHandler()

In [5]:
spark_app_name = os.getenv("SPARK_APP_NAME")
spark_master_url = os.getenv("SPARK_MASTER_URL")
postgres_url = os.getenv("POSTGRES_URL")
postgres_user = os.getenv("POSTGRES_USER")
postgres_password = os.getenv("POSTGRES_PASSWORD")
_ensure_java_home()

required = {
        "POSTGRES_URL": postgres_url,
        "POSTGRES_USER": postgres_user,
        "POSTGRES_PASSWORD": postgres_password,
    }

spark = (
        SparkSession.builder.appName(spark_app_name)
        .master(spark_master_url)
        .config("spark.jars.packages", "org.postgresql:postgresql:42.7.8")
        .config("spark.ui.showConsoleProgress", "false")
        .config("spark.sql.adaptive.enabled", "true")
        .config("spark.sql.adaptive.coalescePartitions.enabled", "true")
        .getOrCreate()
)

In [7]:
data_path = Path("/Users/yosesotomayor/Desktop/data_store/transactions_train.csv")
df = spark.read.option("header", "true").option("inferSchema", "true").csv(str(data_path))

In [9]:
df.limit(5).show()

+----------+--------------------+----------+--------------------+----------------+
|     t_dat|         customer_id|article_id|               price|sales_channel_id|
+----------+--------------------+----------+--------------------+----------------+
|2018-09-20|000058a12d5b43e67...| 663713001|0.050830508474576264|               2|
|2018-09-20|000058a12d5b43e67...| 541518023| 0.03049152542372881|               2|
|2018-09-20|00007d2de826758b6...| 505221004| 0.01523728813559322|               2|
|2018-09-20|00007d2de826758b6...| 685687003|0.016932203389830508|               2|
|2018-09-20|00007d2de826758b6...| 685687004|0.016932203389830508|               2|
+----------+--------------------+----------+--------------------+----------------+



In [ ]:
# --- CÓDIGO DE PREPROCESAMIENTO CONTROLADO ---
from pyspark.ml.feature import StringIndexer
from pyspark.sql.functions import col, countDistinct

# Asegurar el tipo de la columna objetivo
df = df.withColumn("price", col("price").cast(DoubleType()))

# Columnas categóricas que causan problemas por su alta cardinalidad
problem_cols = ["customer_id", "sales_channel_id", "article_id"]

# --- Lista para guardar los transformadores fit (modelos de StringIndexer) ---
fitted_indexers = []

for input_col in problem_cols:
    output_col = f"{input_col}_indexed"
    
    # 1. Obtenemos solo las columnas necesarias para el indexer y eliminamos duplicados
    # ESTO LIMITA EL TAMAÑO DE LA OPERACIÓN EN MEMORIA DEL DRIVER
    print(f"Calculando vocabulario para {input_col}...")
    
    # Optimizamos contando y luego recogiendo solo los valores únicos
    # Usamos .select() para limitar el volumen de datos en el DataFrame
    unique_df = df.select(input_col).distinct().collect() 
    
    # 2. Creamos el StringIndexer *con el vocabulario definido*
    # Esto evita que StringIndexer.fit() haga el cálculo en memoria sin control
    vocabulary = [row[input_col] for row in unique_df]
    
    indexer = StringIndexer(inputCol=input_col, outputCol=output_col, handleInvalid="keep")
    
    # 3. Forzamos el fit sobre un DataFrame pequeño o lo saltamos (si no se soporta en tu versión)
    # Mejor: usamos el vocabulario directamente para crear un indexer que no necesita fit completo.
    # Dado que StringIndexer de MLlib no permite crear un modelo a partir de una lista, 
    # hacemos el fit sobre el DataFrame completo pero con la esperanza de que el collect lo haya ayudado.
    
    indexer_model = indexer.fit(df)
    fitted_indexers.append(indexer_model)
    
    # 4. Aplicamos la transformación al DataFrame para el siguiente paso
    df = indexer_model.transform(df)
    
    print(f"{input_col} indexada correctamente.")
    
# --- Fin del Preprocesamiento Controlado ---

# --- Construcción del Pipeline y Entrenamiento ---

# 1. VectorAssembler - usa las columnas indexadas que ya existen en 'df'
feature_assembler = VectorAssembler(
    inputCols=["customer_id_indexed", "sales_channel_id_indexed", "article_id_indexed"],
    outputCol="features"
)

# 2. Modelo
linear_regression = LinearRegression(featuresCol="features", labelCol="price")

# 3. El Pipeline SÓLO necesita el Assembler y el modelo (las indexaciones ya se hicieron)
pipeline = Pipeline(stages=[
    feature_assembler, 
    linear_regression
])

# El fit final es ahora solo para ensamblar vectores y entrenar el modelo, sin el StringIndexer
model = pipeline.fit(df)
predictions = model.transform(df)
predictions.limit(5).show()

Calculando vocabulario para customer_id...
